# Recipe lab: fine-tuning GPT-2 on Indian recipes, full fine-tuning vs. LoRA



**Problem.** A cooking app wants to draft the method for a dish from its name and ingredient list, in the house style of an Indian recipe website. A pretrained GPT-2 has read a lot of English but writes nothing like a recipe. How much does fine-tuning help, does updating **all** the weights beat training a small **LoRA** adapter, and how do we measure "better"?

You will fine-tune **GPT-2** (124M parameters) on about 3,000 Indian recipes twice, once fully and once with LoRA, under the same data, epochs, and decoding settings. You will compare the original and both fine-tuned models with **BLEU**, **ROUGE**, a simple ingredient check, and your own reading of the outputs. An **optional extension** repeats the LoRA run with **Llama-3.2-1B**.

**Learning objectives**

1. Prepare a recipe dataset for fine-tuning: filter it, and split it by dish so near-duplicate recipes cannot leak from training into evaluation.
2. Turn (prompt, method) pairs into causal-LM training examples whose loss covers only the method and ends with the end-of-text token.
3. Compute BLEU and ROUGE, by hand on a small example and with libraries, and explain what they do and do not measure.
4. Fully fine-tune GPT-2 and fine-tune it with LoRA under the same budget; record trainable parameters, time, and peak memory.
5. Compare the original, fully fine-tuned, and LoRA models on the same validation recipes, including how much each forgets about general English.
6. Merge a LoRA adapter into the base model and check that the outputs do not change.

**Prerequisites:** [Fine-tuning GPT-2](../../lectures/fine_tuning/gpt2_fine_tuning.ipynb) (labels, −100, the end-of-text token, forgetting) · [Fine-tuning with the `Trainer`](../../lectures/fine_tuning/hf_trainer_fine_tuning.ipynb) · [Preparing data](../../lectures/data_preparation/data_preparation.ipynb) (deduplication and group splits) · [PEFT and LoRA](../../lectures/peft/peft_lora.ipynb) · BLEU from the [LSTM encoder–decoder notebook](../../../module2/seq2seq_learning/Seq2Seq_LSTM.ipynb), Section 14. ROUGE is introduced in Exercise 3.

**Data and models**

- **6000+ Indian Food Recipes Dataset** (Jain, 2020), [Mendeley Data, doi:10.17632/xsphgmmh7b.1](https://data.mendeley.com/datasets/xsphgmmh7b/1). 6,871 recipes scraped from Archana's Kitchen, with name, ingredients, method, times, servings, cuisine, course, and diet. Hindi recipes come with machine-translated English columns. License: **CC BY 4.0** (attribution). One CSV file, 24 MB.
- [openai-community/gpt2](https://huggingface.co/openai-community/gpt2): the 124M-parameter GPT-2, MIT license, about 550 MB to download.
- *Optional extension:* [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B): **gated**. You must accept Meta's Llama 3.2 Community License on the model page and use a Hugging Face access token. About 2.5 GB.

**Environment and compute**

- **Kaggle notebook with a GPU (T4)** and **internet on**. Exercises 1–3 also run on a CPU; training GPT-2 on a CPU is possible but too slow for this session.
- Packages: `transformers`, `peft`, `torch`, `pandas` (preinstalled on Kaggle), plus `sacrebleu` and `rouge_score` (install cell below). The notebook was written against `transformers` 5.0 and `peft` 0.21.
- GPU time has not been measured for this exact configuration. On a T4, expect each of the two training runs to take on the order of 5–10 minutes and each evaluation pass about a minute. Record your own times. Plan about **two hours** in total, including the written work; the Llama extension needs about 30 minutes more.
- `TRAIN_LIMIT` in the setup cell limits the training set for a **quick smoke run** that checks the code end to end. A smoke run cannot show what fine-tuning achieves: reset it to `None` for the real run.
- Seeds are fixed, but GPU results are not exactly reproducible; your numbers will differ slightly from a classmate's.

**Files.** Everything is written under `/kaggle/working/recipe_lab/` (or `./recipe_lab_outputs/` elsewhere): the dataset in `data/`, and the LoRA adapter in `adapter/`. Rerunning overwrites the adapter. The models go to the Hugging Face cache. No full model checkpoint is saved.

**How to work:** run the cells in order. Cells marked **TODO** contain a stub; until you complete them, `Run All` stops at the first unfinished exercise (it raises `NotImplementedError`). Most exercises are followed by a check cell.

**Marks (25):** Ex. 1 — 3 · Ex. 2 — 4 · Ex. 3 — 3 · Ex. 4 — 2 · Ex. 5 — 3 · Ex. 6 — 4 · Ex. 7 — 4 · Ex. 8 — 1 · reflection — 1. The Llama extension is not graded.

## Setup

**Installation (Kaggle).** Run the next cell once per session.

**Known Kaggle issue.** Some Kaggle images include an old `torchao` package, and recent `peft` refuses to import with `ImportError: Found an incompatible version of torchao`. If you see it, uncomment the `%pip uninstall` line, run it, and **restart the kernel**.

In [ ]:
# %pip install -q sacrebleu rouge_score
# %pip uninstall -y torchao          # only if peft raises "Found an incompatible version of torchao"

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle "T4 x2": train on one GPU only

import hashlib
import math
import random
import re
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
GB = 1e9
CKPT = "openai-community/gpt2"
MAX_LEN = 768            # longest training example in tokens; longer ones are dropped, not truncated
MAX_NEW = 400            # longest generated method, in tokens
N_EVAL = 40              # validation recipes for BLEU/ROUGE and reading
EPOCHS = 2
BATCH = 8
TRAIN_LIMIT = None       # e.g. 200 for a quick smoke run; None for the real run
OUT = Path("/kaggle/working/recipe_lab") if Path("/kaggle/working").is_dir() else Path("recipe_lab_outputs")
DATA_DIR, ADAPTER_DIR = OUT / "data", OUT / "adapter"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (Exercises 1-3 only)")

## The data

The next cell downloads the dataset's CSV file (24 MB) from Mendeley Data, once, and checks its SHA-256 hash, so everyone works with the same file. If your environment has no internet, download `IndianFoodDatasetCSV.csv` from the [dataset page](https://data.mendeley.com/datasets/xsphgmmh7b/1) and put it in the `data/` folder printed below.

In [ ]:
DATA_URL = ("https://data.mendeley.com/public-files/datasets/xsphgmmh7b/files/"
            "6fb5ae32-a815-4b8f-9d68-8d512227eac8/file_downloaded")
DATA_SHA256 = "273f8d45f41aca9af0396dda07af137c68c7387a1506992904144b944c4346be"
csv_path = DATA_DIR / "IndianFoodDatasetCSV.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not csv_path.exists():
    print("downloading to", csv_path)
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "recipe-lab"})   # the default agent gets 403
    with urllib.request.urlopen(request) as response:
        csv_path.write_bytes(response.read())
assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == DATA_SHA256, "unexpected file: delete it and download again"

raw = pd.read_csv(csv_path, encoding="utf-8-sig")
raw = raw.apply(lambda col: col if pd.api.types.is_numeric_dtype(col) else col.str.replace("\ufeff", "").str.strip())
print(raw.shape)
print(raw.columns.tolist())
raw.iloc[0][["RecipeName", "Cuisine", "Course", "Diet", "Ingredients", "Instructions"]].to_dict()

**Reading the output.** Each row is one web page of the recipe site. `Ingredients` is one comma-separated string; `Instructions` is the method as one paragraph. About a fifth of the pages are in Hindi; for those, the `Translated…` columns hold a machine translation (made with Google Translate by the dataset's author). Several dishes appear twice: once in English and once as "(Recipe In Hindi)".

The next cell applies three filters. They are given, not part of an exercise, but read them: each is a data decision you should be able to justify.

1. **English originals only.** Drop rows whose original name, ingredients, or method contain Devanagari script. This removes the Hindi pages and their machine translations, which are of uneven quality and are often a second copy of an English recipe.
2. **Indian cuisines only.** The site also publishes Continental, Italian, Chinese, Mexican and other recipes, and a few rows have a course ("Snack", "Dinner") in the cuisine field. Drop those.
3. **Cleaning.** Add the missing space in run-together sentences ("Keep aside.Heat oil"), remove the word "Recipe" from dish names, and split the ingredient string into a list.

In [ ]:
DEVANAGARI = r"[\u0900-\u097F]"
NOT_INDIAN = {"Afghan", "African", "American", "Arab", "Asian", "British", "Burmese", "Cantonese", "Caribbean",
              "Chinese", "Continental", "European", "French", "Fusion", "Greek", "Hunan", "Indonesian",
              "Italian Recipes", "Japanese", "Jewish", "Korean", "Malaysian", "Mediterranean", "Mexican",
              "Middle Eastern", "Nepalese", "Pakistani", "Shandong", "Sichuan", "Sri Lankan", "Thai",
              "Vietnamese", "World Breakfast",
              "Appetizer", "Brunch", "Dessert", "Dinner", "Lunch", "Side Dish", "Snack"}   # courses, not cuisines


def clean_text(t):
    t = re.sub(r"\.(?=[A-Z])", ". ", t)        # "aside.Heat" -> "aside. Heat"
    t = re.sub(r",(?=[A-Za-z])", ", ", t)       # "Recipe,de-seed" -> "Recipe, de-seed"
    return " ".join(t.split())


def clean_name(name):
    return " ".join(re.sub(r"\brecipe\b", "", name, flags=re.IGNORECASE).split()).strip(" -")


english = ~(raw["RecipeName"].str.contains(DEVANAGARI) | raw["Ingredients"].fillna("").str.contains(DEVANAGARI)
            | raw["Instructions"].str.contains(DEVANAGARI))
kept = raw[english & raw["Ingredients"].notna() & ~raw["Cuisine"].isin(NOT_INDIAN)]
recipes = [{"raw_name": r.RecipeName, "name": clean_name(r.RecipeName), "cuisine": r.Cuisine,
            "course": r.Course, "diet": r.Diet,
            "ingredients": [i.strip() for i in r.Ingredients.split(",") if i.strip()],
            "method": clean_text(r.Instructions)} for r in kept.itertuples()]
print(f"{len(raw):,} rows -> {english.sum():,} English originals -> {len(recipes):,} Indian recipes")
print(kept["Cuisine"].value_counts().head(8).to_string())

## Exercise 1 — Split by dish, not by row (3 marks)

The site often has two pages for one dish: "Aamras Recipe" and "Aamras Recipe - Ripe Mango Ras", or the same name published twice. Their methods share many phrases. If one copy lands in training and the other in validation, the fine-tuned model can score well on validation by **remembering** a recipe it was trained on, which inflates BLEU and ROUGE. (This is the leakage problem from the data-preparation lecture.)

The fix is a **group split**: give every recipe a *dish key*, and put all recipes with the same key in the same split.

Complete `dish_key(raw_name)`:

1. Lowercase the name.
2. Keep only the part before the first `" - "` or `"("`.
3. Remove the word `recipe` (or `recipes`) as a whole word.
4. Replace every character that is not a letter `a`–`z` or a space with a space, and collapse repeated spaces (strip both ends).

In [ ]:
def dish_key(raw_name):
    # TODO: steps 1-4 above (re.split, re.sub, and " ".join(s.split()) are useful)
    raise NotImplementedError("Exercise 1: complete dish_key")

In [ ]:
assert dish_key("Aamras Recipe - Ripe Mango Ras") == dish_key("Aamras Recipe") == "aamras"
assert dish_key("Achari Paneer Masala Recipe") == "achari paneer masala"
assert dish_key("Bengali Doi Maach Recipe (Fish In Yogurt Curry)") == "bengali doi maach"
for r in recipes:
    r["key"] = dish_key(r["raw_name"])

# Group split: shuffle the distinct keys, then 80% / 10% / 10% of the keys -> train / validation / test
keys = sorted({r["key"] for r in recipes})
random.Random(SEED).shuffle(keys)
n_val = n_test = len(keys) // 10
split_of = {k: "val" for k in keys[:n_val]} | {k: "test" for k in keys[n_val:n_val + n_test]} \
         | {k: "train" for k in keys[n_val + n_test:]}
splits = {s: [r for r in recipes if split_of[r["key"]] == s] for s in ("train", "val", "test")}
assert not ({r["key"] for r in splits["train"]} & {r["key"] for r in splits["val"] + splits["test"]})
print({s: len(rs) for s, rs in splits.items()}, f"| {len(keys):,} distinct dishes")

# What a random *row* split would have done
shuffled = random.Random(SEED).sample(recipes, len(recipes))
row_val, row_train = shuffled[:len(recipes) // 10], shuffled[len(recipes) // 10:]
train_keys = {r["key"] for r in row_train}
twins = [r for r in row_val if r["key"] in train_keys]
print(f"random row split: {len(twins)} of {len(row_val)} validation recipes have a same-dish twin in training")
for r in twins[:3]:
    print("  ", r["raw_name"], "<->", next(t["raw_name"] for t in row_train if t["key"] == r["key"]))
print("Exercise 1 checks passed.")

**Question 1.** (a) Use the printed numbers: how many validation recipes would a random row split have leaked, and why does even a small number matter for BLEU/ROUGE? (b) Give one pair of recipes that `dish_key` would **not** group although they are really the same dish, and one pair it groups although they are different. Is the key too strict or too loose? (c) The test split is created but never used in this lab. Why keep it?

**Your answer — Question 1:** *(write here)*

## The prompt format

Every example is a **prompt** (what the model is given) followed by the **method** (what it should write). GPT-2 is a plain language model, not a chat model, so the prompt is plain text with field labels. The same format is used for training, for the original model, and for the fine-tuned models.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CKPT)     # a few MB; the model is downloaded in Exercise 4
tokenizer.pad_token = tokenizer.eos_token           # GPT-2 has no pad token; padding is masked anyway


def format_prompt(r):
    ingredients = "\n".join(f"- {i}" for i in r["ingredients"])
    return (f"Dish: {r['name']}\nCuisine: {r['cuisine']} | Course: {r['course']} | Diet: {r['diet']}\n"
            f"Ingredients:\n{ingredients}\nMethod:\n")


def prompt_ids(r, tok, add_bos):
    """Token IDs of the prompt. GPT-2 uses no beginning-of-text token; Llama does (see the extension)."""
    ids = tok(format_prompt(r), add_special_tokens=False)["input_ids"]
    return ([tok.bos_token_id] if add_bos else []) + ids


r0 = splits["train"][0]
print(format_prompt(r0) + r0["method"][:300] + " …")
print("\nend-of-text token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
for word in ["potato", "cumin", "karela", "amchur", "kadhai", "tadka"]:
    print(f"{word:>8} -> {tokenizer.tokenize(' ' + word)}")

The last lines show how GPT-2's byte-pair encoding splits words. Common English words are one token; Indian ingredient and utensil names, rare in GPT-2's training text, are split into several pieces. Every extra piece costs context length and is one more prediction the model can get wrong.

## Exercise 2 — Training examples with a masked prompt (4 marks)

As in the [GPT-2 fine-tuning notebook](../../lectures/fine_tuning/gpt2_fine_tuning.ipynb), we want the loss to cover **only the method**, and the method must end with the end-of-text token so that the model learns to stop. Positions whose label is **−100** are ignored by the loss, and the model shifts the labels by one position internally.

Complete `build_example(r, tok, add_bos)`:

1. `p` = `prompt_ids(r, tok, add_bos)`.
2. `a` = the tokens of `r["method"]` (with `add_special_tokens=False`), followed by `tok.eos_token_id`.
3. `input_ids` = `p + a`; `attention_mask` = all ones; `labels` = −100 for every prompt position, followed by `a`.

Return a dict of plain Python lists under `input_ids`, `attention_mask`, and `labels`. The function takes the tokenizer as an argument so that the Llama extension can reuse it.

In [ ]:
def build_example(r, tok, add_bos):
    # TODO: steps 1-3 above
    raise NotImplementedError("Exercise 2: complete build_example")

In [ ]:
ex = build_example(r0, tokenizer, add_bos=False)
n_prompt = len(prompt_ids(r0, tokenizer, add_bos=False))
assert len(ex["input_ids"]) == len(ex["labels"]) == len(ex["attention_mask"])
assert all(l == -100 for l in ex["labels"][:n_prompt]), "every prompt position must be -100"
assert ex["labels"][n_prompt:] == ex["input_ids"][n_prompt:], "method labels must equal the method tokens"
assert ex["input_ids"][-1] == tokenizer.eos_token_id, "the method must end with the end-of-text token"
assert tokenizer.decode(ex["input_ids"][:n_prompt], clean_up_tokenization_spaces=False) == format_prompt(r0)
print(f"{len(ex['input_ids'])} tokens, {len(ex['input_ids']) - n_prompt} of them scored")
print("scored text ends with:", repr(tokenizer.decode([t for t in ex["labels"] if t != -100])[-80:]))


def make_dataset(rows, tok, add_bos):
    data = [build_example(r, tok, add_bos) for r in rows]
    return [e for e in data if len(e["input_ids"]) <= MAX_LEN]


train_rows = splits["train"][:TRAIN_LIMIT]
train_data = make_dataset(train_rows, tokenizer, add_bos=False)
val_data = make_dataset(splits["val"], tokenizer, add_bos=False)
eval_rows = random.Random(SEED).sample(splits["val"], N_EVAL)     # for generation and BLEU/ROUGE
lengths = sorted(len(build_example(r, tokenizer, False)["input_ids"]) for r in train_rows)
print(f"training examples: {len(train_data):,} kept of {len(train_rows):,} (> {MAX_LEN} tokens dropped)"
      f" | validation examples: {len(val_data)}")
print(f"tokens per example: median {lengths[len(lengths) // 2]}, 90th percentile {lengths[int(0.9 * len(lengths))]}, "
      f"max {lengths[-1]}")
print("Exercise 2 checks passed.")

**Question 2.** (a) Why is the prompt masked? Which part of the prompt would the model waste effort learning if it were not? (b) What does the end-of-text token at the end of the labels teach the model? What would you expect to see at generation time without it? (c) Why drop examples longer than `MAX_LEN` instead of truncating them? Use the printed length statistics to say how many examples this costs.

**Your answer — Question 2:** *(write here)*

## Exercise 3 — BLEU and ROUGE (3 marks)

Both metrics compare a generated text, the **candidate**, with a human-written **reference** by counting shared words and word sequences (*n-grams*). Neither looks at meaning.

**BLEU** (Papineni et al., 2002), recapped from Module 2: for n = 1 to 4, the **clipped n-gram precision** $p_n$ is the fraction of the candidate's n-grams that also occur in the reference, where each reference n-gram can be matched at most as many times as it occurs there. BLEU is the geometric mean of $p_1 \dots p_4$ times a **brevity penalty** $\mathrm{BP} = \min(1, e^{1 - r/c})$, where $c$ is the candidate length and $r$ the reference length. Precision rewards saying only what the reference says; BP stops a short candidate from winning by saying very little. **Corpus BLEU** adds up the n-gram counts over all candidates before dividing, so it is not the average of per-sentence BLEU scores. We use `sacrebleu`, which fixes the tokenization so that scores are comparable (Post, 2018), and report BLEU on a 0–100 scale.

**ROUGE** (Lin, 2004) was designed for summaries, where the question is how much of the reference the candidate **recovers**:

- **ROUGE-1** and **ROUGE-2** count overlapping unigrams and bigrams. Recall = overlap / reference n-grams; precision = overlap / candidate n-grams; we report their harmonic mean, **F1**.
- **ROUGE-L** uses the **longest common subsequence** (LCS): the longest sequence of words that appears in both texts in the same order, not necessarily next to each other. Recall = LCS / reference length, precision = LCS / candidate length, then F1.

We use Google's `rouge_score` package with stemming, and average F1 over the recipes (× 100).

**(a) By hand.** Candidate: *heat oil in a pan*. Reference: *heat the oil in a kadhai*. Compute ROUGE-1 precision, recall, and F1; ROUGE-2 F1; the LCS and ROUGE-L F1; and BLEU's brevity penalty for this pair. Write your working in the answer cell below; the check cell compares your F1 values with the library's.

**(b) Code.** Complete `text_metrics(preds, refs)`. It returns a dict with `"BLEU"` (corpus BLEU from `sacrebleu.corpus_bleu(preds, [refs]).score`) and `"ROUGE-1"`, `"ROUGE-2"`, `"ROUGE-L"` (mean F1 × 100, using `scorer.score(reference, prediction)`: **reference first**).

**Your answer — Question 3(a):** *(write here)*

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer

ROUGE_KEYS = {"ROUGE-1": "rouge1", "ROUGE-2": "rouge2", "ROUGE-L": "rougeL"}
scorer = rouge_scorer.RougeScorer(list(ROUGE_KEYS.values()), use_stemmer=True)


def text_metrics(preds, refs):
    # TODO: corpus BLEU, and mean F1 x 100 for each ROUGE variant
    raise NotImplementedError("Exercise 3: complete text_metrics")

In [ ]:
m = text_metrics(["heat oil in a pan"], ["heat the oil in a kadhai"])
print({k: round(v, 1) for k, v in m.items()})
assert abs(m["ROUGE-1"] - 72.7) < 0.1 and abs(m["ROUGE-2"] - 44.4) < 0.1 and abs(m["ROUGE-L"] - 72.7) < 0.1
assert abs(text_metrics(["add salt and stir well"], ["add salt and stir well"])["BLEU"] - 100) < 1e-6
print("Exercise 3 checks passed.")

BLEU and ROUGE only count words. They cannot tell whether a method actually uses the ingredients it was given, so we add one task-specific check. `ingredient_coverage` reduces each listed ingredient to its last word ("Red Chilli powder" → *powder*, "Sunflower Oil - as required" → *oil*) and reports the fraction of these words that appear in the method. It is deliberately crude: it misses synonyms (*besan* vs. *gram flour*) and cannot tell *using* an ingredient from *mentioning* it. We apply it to the human-written references too, which gives a realistic upper range.

In [ ]:
UNITS = r"\b(cups?|tablespoons?|teaspoons?|tbsp|tsp|grams?|gms?|g|kg|ml|litres?|liters?|inch|pinch|sprigs?|cloves?|pieces?|nos?)\b"


def ingredient_words(ingredients):
    words = set()
    for item in ingredients:
        name = re.split(r" - |\(", item.lower())[0]
        name = re.sub(UNITS, " ", re.sub(r"[^a-z ]", " ", name))
        if name.split():
            words.add(name.split()[-1])
    return words


def ingredient_coverage(rows, methods):
    fractions = []
    for r, m in zip(rows, methods):
        words, text = ingredient_words(r["ingredients"]), set(re.findall(r"[a-z]+", m.lower()))
        fractions.append(len(words & text) / max(len(words), 1))
    return 100 * sum(fractions) / len(fractions)


print(sorted(ingredient_words(r0["ingredients"])))
print(f"reference methods mention {ingredient_coverage(eval_rows, [r['method'] for r in eval_rows]):.0f}% "
      "of their ingredient words (validation sample)")

## Exercise 4 — Baseline: the original GPT-2 (2 marks)

> **This downloads GPT-2 (about 550 MB) and should run on the GPU.**

The helper below generates a method for each recipe with **greedy decoding** (deterministic, no sampling), at most `MAX_NEW` tokens, and `no_repeat_ngram_size=4`, which forbids repeating any 4-token sequence. Without that rule, greedy GPT-2 quickly loops ("add the onions and add the onions and …"). The rule is a decoding choice, and it is kept **fixed for every model** in the lab so that the comparison stays fair.

Prompts are **left-padded** in a batch, so that every prompt ends at the same position and generation continues from there. The helper also records whether each output **stopped** by emitting the end-of-text token, or ran out of tokens.

In [ ]:
from transformers import AutoModelForCausalLM


@torch.no_grad()
def generate_methods(model, tok, rows, add_bos, batch_size=8, max_new_tokens=MAX_NEW):
    model.eval()
    texts, stopped = [], []
    for i in range(0, len(rows), batch_size):
        prompts = [prompt_ids(r, tok, add_bos) for r in rows[i:i + batch_size]]
        width = max(len(p) for p in prompts)
        ids = torch.full((len(prompts), width), tok.pad_token_id)
        mask = torch.zeros((len(prompts), width), dtype=torch.long)
        for j, p in enumerate(prompts):
            ids[j, width - len(p):] = torch.tensor(p)
            mask[j, width - len(p):] = 1
        out = model.generate(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE), do_sample=False,
                             temperature=None, top_p=None, top_k=None, no_repeat_ngram_size=4,
                             max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
        for new in out[:, width:].tolist():
            stop = tok.eos_token_id in new
            new = new[:new.index(tok.eos_token_id)] if stop else new
            texts.append(tok.decode(new, skip_special_tokens=True).strip())
            stopped.append(stop)
    return texts, stopped


refs = [r["method"] for r in eval_rows]
results = {}                                        # model name -> metrics row


def evaluate(name, preds, stopped, rows=None):
    rows = eval_rows if rows is None else rows
    row = text_metrics(preds, [r["method"] for r in rows])
    row["ingredients %"] = ingredient_coverage(rows, preds)
    row["words"] = sum(len(p.split()) for p in preds) / len(preds)
    row["stopped %"] = 100 * sum(stopped) / len(stopped)
    results[name] = row
    return pd.DataFrame(results).T.round(1)


transformers.set_seed(SEED)
base_model = AutoModelForCausalLM.from_pretrained(CKPT).to(DEVICE)
start = time.perf_counter()
base_preds, base_stopped = generate_methods(base_model, tokenizer, eval_rows, add_bos=False)
print(f"generated {len(base_preds)} methods in {time.perf_counter() - start:.0f} s")
reference_row = {"ingredients %": ingredient_coverage(eval_rows, refs), "words": sum(len(t.split()) for t in refs) / len(refs)}
print("references:", {k: round(v, 1) for k, v in reference_row.items()})
evaluate("GPT-2 original", base_preds, base_stopped)

In [ ]:
for r, p in list(zip(eval_rows, base_preds))[:2]:
    print(f"=== {r['name']} ===\nMODEL:     {p[:600]}\nREFERENCE: {r['method'][:600]}\n")

**Question 4.** Read at least five baseline outputs. (a) Describe what the original GPT-2 writes after `Method:`, with one short quotation. (b) The baseline's BLEU and ROUGE are not zero. Which words and n-grams can it share with a reference without having learned anything about cooking?

**Your answer — Question 4:** *(write here)*

## Exercise 5 — Full fine-tuning (3 marks)

Full fine-tuning updates all of GPT-2's weights. Complete `make_args(name, lr)`, which returns the `TrainingArguments` for a run; Exercise 6 reuses it so that both runs get the same budget. Use:

- output directory `str(OUT / name)`; batch size `BATCH` for training and evaluation; `EPOCHS` epochs;
- learning rate `lr` with a cosine schedule and 20 warm-up steps;
- fp16 mixed precision **only if a GPU is available** (`fp16=HAS_GPU`);
- evaluation and logging: evaluate at the end of every epoch (`eval_strategy="epoch"`), log every 20 steps;
- `save_strategy="no"` (no checkpoints), `report_to="none"`, `remove_unused_columns=False`, and `seed=SEED`.

The cell after it, which trains, is given.

In [ ]:
from transformers import Trainer, TrainingArguments

FULL_LR = 5e-5


def make_args(name, lr):
    # TODO: return TrainingArguments(...) as described above
    raise NotImplementedError("Exercise 5: complete make_args")

The collator pads each batch to its longest example on the right: input IDs with the pad token, labels with −100 (padding is never scored), attention mask with 0. `train` runs the `Trainer`, measures time and peak GPU memory, and plots the losses. With `fp16=True`, the `Trainer` runs the model's forward pass under mixed-precision *autocast* and leaves that wrapper on the model after training; `train` removes it, so that every model in this lab, trained or not, is evaluated in the same precision (fp32).

> **The next cell trains GPT-2.** Record the time and memory it prints.

In [ ]:
from accelerate.utils import extract_model_from_parallel


def collate(batch, pad_id=tokenizer.pad_token_id):
    width = max(len(e["input_ids"]) for e in batch)
    input_ids = torch.full((len(batch), width), pad_id)
    labels = torch.full((len(batch), width), -100)
    attention_mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, e in enumerate(batch):
        n = len(e["input_ids"])
        input_ids[i, :n] = torch.tensor(e["input_ids"])
        labels[i, :n] = torch.tensor(e["labels"])
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train(model, args, train_data, eval_data, title, data_collator=collate):
    trainer = Trainer(model=model, args=args, train_dataset=train_data, eval_dataset=eval_data,
                      data_collator=data_collator)
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    extract_model_from_parallel(model, keep_fp32_wrapper=False)   # remove the fp16 autocast wrapper: evaluate in fp32
    stats = {"trainable": count_trainable(model), "minutes": (time.perf_counter() - start) / 60,
             "peak GB": torch.cuda.max_memory_allocated() / GB if HAS_GPU else float("nan")}
    log = trainer.state.log_history
    tr = [(h["step"], h["loss"]) for h in log if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in log if "eval_loss" in h]
    plt.figure(figsize=(6, 3))
    if tr:
        plt.plot(*zip(*tr), marker=".", label="training loss")
    plt.plot(*zip(*ev), marker="o", label="validation loss")
    plt.xlabel("optimizer step"); plt.ylabel("loss (method tokens only)"); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()
    print({k: (f"{v:,}" if k == "trainable" else round(v, 2)) for k, v in stats.items()},
          "| final validation loss", round(ev[-1][1], 3))
    return stats, ev[-1][1]


transformers.set_seed(SEED)
full_model = AutoModelForCausalLM.from_pretrained(CKPT).to(DEVICE)
full_stats, full_val_loss = train(full_model, make_args("full", FULL_LR), train_data, val_data, "Full fine-tuning")
assert full_stats["trainable"] == 124_439_808, "full fine-tuning trains every GPT-2 parameter"
print("Exercise 5 checks passed.")

**Question 5.** (a) Report the training time, peak memory, and final validation loss. (b) Use the bytes-per-parameter accounting from the [PEFT lecture](../../lectures/peft/peft_lora.ipynb) (weights, gradients, and two Adam moments in fp32: about 16 bytes per trainable parameter) to estimate the memory for GPT-2's weights and optimizer. Is the measured peak higher or lower, and what else uses memory during training? (c) The validation loss is computed on held-out dishes. Why is a falling validation loss still not enough to say the model writes better methods?

**Your answer — Question 5:** *(write here)*

## Exercise 6 — LoRA (4 marks)

Now freeze GPT-2 and train only a LoRA adapter on the attention's input projection. In GPT-2 this is one layer per block, `c_attn`, which maps the 768-dimensional hidden state to the concatenated query, key, and value vectors (768 × 3 = 2,304 outputs). GPT-2 has 12 blocks.

**(a) Predict before you build.** With rank `r = 8`, how many trainable parameters does LoRA add? Recall that LoRA adds $A \in \mathbb{R}^{r \times d_\text{in}}$ and $B \in \mathbb{R}^{d_\text{out} \times r}$ to each adapted layer. Set `predicted_lora_params` to your number (as an arithmetic expression, not a copied total).

**(b) Configure.** Create `lora_config`: rank 8, `lora_alpha` 16, dropout 0.05, `target_modules=["c_attn"]`, task type causal LM, and `fan_in_fan_out=True`. (GPT-2 implements its linear layers as `Conv1D` modules, which store the weight transposed, as (in, out); this flag tells `peft` so. Without it, `peft` sets it for you and prints a warning.) Then wrap a **freshly loaded** GPT-2 with `get_peft_model`.

LoRA usually needs a higher learning rate than full fine-tuning, because only the small adapter moves and its update starts at zero (`B = 0`). We use `LORA_LR = 5e-4`, ten times `FULL_LR`; neither value was tuned for this lab.

In [ ]:
from peft import LoraConfig, get_peft_model

LORA_LR = 5e-4
transformers.set_seed(SEED)
lora_base = AutoModelForCausalLM.from_pretrained(CKPT).to(DEVICE)
# TODO (a): predicted_lora_params = ...
# TODO (b): lora_config = LoraConfig(...)
# TODO (b): lora_model = get_peft_model(lora_base, lora_config)
raise NotImplementedError("Exercise 6: predict the count, create lora_config and lora_model")

In [ ]:
lora_model.print_trainable_parameters()
assert count_trainable(lora_model) == predicted_lora_params, "your prediction does not match peft's count"
assert predicted_lora_params == 294_912, "check the shapes of A and B"
print("Exercise 6 configuration checks passed.")

> **The next cell trains the LoRA adapter**, with the same data, epochs, batch size, and schedule as Exercise 5.

In [ ]:
lora_stats, lora_val_loss = train(lora_model, make_args("lora", LORA_LR), train_data, val_data, "LoRA (r = 8, c_attn)")
pd.DataFrame({"full fine-tuning": full_stats | {"val loss": full_val_loss},
              "LoRA": lora_stats | {"val loss": lora_val_loss}}).T

**Question 6.** (a) Show your calculation for `predicted_lora_params`, and give the adapter's size as a percentage of GPT-2's parameters. (b) Compare the two runs' training time and peak memory. LoRA trains about 0.24% of the parameters: why is the memory saving much smaller than that, and why is training not 400× faster? (c) Compare the two validation losses. Which run fits the recipes' wording better, and is that what you expected?

**Your answer — Question 6:** *(write here)*

## Exercise 7 — Compare the three models (4 marks)

Generate methods for the same validation recipes with both fine-tuned models, using the same helper and settings as the baseline.

In [ ]:
start = time.perf_counter()
full_preds, full_stopped = generate_methods(full_model, tokenizer, eval_rows, add_bos=False)
lora_preds, lora_stopped = generate_methods(lora_model, tokenizer, eval_rows, add_bos=False)
print(f"generated {2 * len(eval_rows)} methods in {time.perf_counter() - start:.0f} s")
evaluate("GPT-2 full FT", full_preds, full_stopped)
table = evaluate("GPT-2 LoRA", lora_preds, lora_stopped)
table

In [ ]:
for i in (0, 1, 2):
    r = eval_rows[i]
    print(f"=================== {r['name']} ===================")
    print("INGREDIENTS:", "; ".join(r["ingredients"]))
    for name, preds in (("REFERENCE", refs), ("FULL FT", full_preds), ("LORA", lora_preds)):
        print(f"\n{name}: {preds[i][:700]}")
    print()

**Forgetting.** Fine-tuning on recipes can make a model worse at ordinary English, as in the lecture's Section 7. Measure it: complete `mean_loss(model, texts)`, the **token-weighted** mean cross-entropy of a model on a list of texts. For each text, tokenize it, call the model with `labels=input_ids` (the model shifts them), and weight its `.loss` by the number of predicted tokens, which is one less than the number of tokens. Use `torch.no_grad()` and `model.eval()`.

The four paragraphs below were written for this lab, about everyday topics with no cooking.

In [ ]:
GENERAL_TEXTS = [
    "The train to Jaipur left the station twenty minutes late. Most passengers had already found their seats, "
    "and a few were reading newspapers while the countryside moved slowly past the windows.",
    "Rainwater that falls on the hills collects in small streams, which join to form rivers. Some of the water "
    "soaks into the ground and is stored in aquifers that supply wells during the dry season.",
    "The committee will meet on Monday to discuss the budget for the next financial year. Members are asked to "
    "read the report in advance and to send any questions to the secretary by Friday.",
    "In the final over, the batsman needed twelve runs to win. He hit the first ball for four, missed the next "
    "two, and then lifted a full toss over long-on for six.",
]

In [ ]:
@torch.no_grad()
def mean_loss(model, texts):
    # TODO: token-weighted mean loss over the texts
    raise NotImplementedError("Exercise 7: complete mean_loss")

In [ ]:
forgetting = {name: mean_loss(m, GENERAL_TEXTS) for name, m in
              (("GPT-2 original", base_model), ("GPT-2 full FT", full_model), ("GPT-2 LoRA", lora_model))}
for name, loss in forgetting.items():
    print(f"{name:<16} loss on general English {loss:.3f}  (perplexity {math.exp(loss):.1f})")
with lora_model.disable_adapter():                    # the same object without its adapter = the original GPT-2
    assert abs(mean_loss(lora_model, GENERAL_TEXTS) - forgetting["GPT-2 original"]) < 1e-3
print("Exercise 7 checks passed.")

**Question 7.** (a) Summarize the table: how did BLEU, ROUGE, ingredient coverage, length, and "stopped %" change from the original model to each fine-tuned model? (b) Most reference methods begin "To begin making the … Recipe," and end with a serving suggestion. How could this house style affect BLEU and ROUGE, and what does that tell you about what the scores reward? (c) For the three printed recipes, judge each fine-tuned method yourself: does it use the listed ingredients, are the steps in a sensible order, and does it invent an ingredient or an unsafe instruction? Does your judgment agree with the metrics? (d) Which model forgot more general English? Explain why in terms of what each method changes.

**Your answer — Question 7:** *(write here)*

## Exercise 8 — Save and merge the adapter (1 mark)

Only the adapter is saved: a user who has GPT-2 needs nothing else. For deployment, the adapter can be **merged** into the weights ($W \leftarrow W + \frac{\alpha}{r} BA$), which removes LoRA's extra computation. Merging should not change the outputs, up to floating-point rounding.

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(ADAPTER_DIR)
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.iterdir()) / 1e6
print(f"adapter on disk: {adapter_mb:.2f} MB | GPT-2 in fp32: {124_439_808 * 4 / 1e6:.0f} MB")

probe = tokenizer(format_prompt(eval_rows[0]), return_tensors="pt").to(DEVICE)
with torch.no_grad():
    before = lora_model(**probe).logits
    merged = lora_model.merge_and_unload()           # modifies the model in place: run this cell last
    after = merged(**probe).logits
print(f"largest logit change after merging: {(before - after).abs().max().item():.2e}")
assert (before - after).abs().max().item() < 1e-2, "merging should change the logits only by rounding"
assert sum(p.numel() for p in merged.parameters()) == 124_439_808
assert not any("lora" in n for n, _ in merged.named_parameters())

**Question 8.** (a) Explain the adapter's size from its parameter count. (b) Why is the largest logit change not exactly zero? (c) A team wants five recipe styles (by region) on one server. Would you merge five adapters into five model copies, or keep one base model and swap adapters? Give one reason for each choice.

**Your answer — Question 8:** *(write here)*

## Reflection and deliverables (1 mark)

Submit this notebook **with outputs**, your answers to Questions 1–8, and a short paragraph (about 150 words):

*The cooking app has a GPU budget for one fine-tuned GPT-2. Would you choose full fine-tuning or LoRA? Base your recommendation on the evidence you collected (the metrics table, your reading of the outputs, forgetting, time, memory, storage), say what that evidence does not show, and name the next experiment you would run.*

**Responsible use.**

- **Food safety.** A fluent method can still be wrong about cooking times, temperatures, or ingredients (raw kidney beans, undercooked meat, allergens not in the ingredient list). Generated recipes need human review before anyone cooks from them, and BLEU/ROUGE do not measure safety.
- **Licensing and attribution.** The dataset is CC BY 4.0, which requires attribution to its author. The recipe texts themselves were scraped from Archana's Kitchen, whose own terms apply to that content; use the data for study, and check the terms before publishing a model or its outputs. GPT-2 is MIT-licensed.
- **Memorization.** A fine-tuned model can reproduce training recipes word for word. Before publishing outputs as "new" recipes, check them against the training data.
- **Representation.** The site covers some regions far more than others (see the cuisine counts). A model fine-tuned on it will write North and South Indian dishes more fluently than, say, North-Eastern ones.

**Optional extensions (not graded).** (1) Train LoRA with `r` = 2 and 32 and plot validation loss against `r` (lecture, Section 6). (2) Add `c_proj` and the MLP layers (`c_fc`, `mlp.c_proj`) to `target_modules` and compare. (3) Recompute BLEU/ROUGE without the first sentence of each method (Question 7b).

## Optional extension — LoRA on Llama-3.2-1B (not graded)

GPT-2 dates from 2019. Does a modern model of about ten times its size write better methods after the same kind of LoRA fine-tuning? Llama-3.2-1B uses the components from Module 2's [modern LLM](../../../module2/transformers/attention_mechanisms.ipynb) unit: rotary position embeddings, grouped-query attention (32 query heads but only 8 key/value heads), and a 128,256-token vocabulary. We use the **base** model, not the Instruct version, so that it gets the same plain-text prompt as GPT-2.

**Access.** The model is gated. Once per account:

1. Sign in to Hugging Face, open [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B), read the Llama 3.2 Community License and Acceptable Use Policy, and request access. Approval is usually quick, but not instant.
2. Create an access token with read permission (Settings → Access Tokens).
3. In Kaggle: Add-ons → Secrets → add a secret named `HF_TOKEN` with the token, and attach it to this notebook. Elsewhere, set the environment variable `HF_TOKEN`. **Never paste the token into a cell.**

**Compute.** The model is about 2.5 GB in 16-bit and is trained in 16-bit with LoRA (no quantization). To keep the run short, it trains for one epoch on `LLAMA_N_TRAIN` recipes and is evaluated on the first `LLAMA_N_EVAL` evaluation recipes. Time and memory have not been measured; expect roughly 10–20 minutes of training on a T4. Set `RUN_LLAMA = True` to run the section; the cell below first frees the GPT-2 models.

In [ ]:
RUN_LLAMA = False
LLAMA_CKPT = "meta-llama/Llama-3.2-1B"
LLAMA_N_TRAIN, LLAMA_N_EVAL = 1000, 20


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")


if RUN_LLAMA:
    assert HAS_GPU, "the Llama extension needs a GPU"
    import gc
    del base_model, full_model, lora_model, lora_base, merged
    gc.collect(); torch.cuda.empty_cache()
    HF_TOKEN = hf_token()
    assert HF_TOKEN, "no Hugging Face token found: see 'Access' above"
    llama_tok = AutoTokenizer.from_pretrained(LLAMA_CKPT, token=HF_TOKEN)
    if llama_tok.pad_token is None:
        llama_tok.pad_token = llama_tok.eos_token
    print("BOS", repr(llama_tok.bos_token), llama_tok.bos_token_id, "| EOS", repr(llama_tok.eos_token), llama_tok.eos_token_id)
    for word in ["potato", "cumin", "karela", "amchur", "kadhai", "tadka"]:
        print(f"{word:>8} -> {llama_tok.tokenize(' ' + word)}")
    sample = splits["train"][:200]
    n_gpt2 = sum(len(build_example(r, tokenizer, False)["input_ids"]) for r in sample)
    n_llama = sum(len(build_example(r, llama_tok, True)["input_ids"]) for r in sample)
    print(f"tokens for 200 training recipes: GPT-2 {n_gpt2:,} | Llama {n_llama:,} ({n_llama / n_gpt2:.2f}x)")

`build_example` is reused with `add_bos=True`: Llama was trained with `<|begin_of_text|>` at the start of every document, and we add it **explicitly** rather than relying on the tokenizer's default, which has changed between library versions. The check below prints the first and last tokens, as you should always do with a new tokenizer.

In [ ]:
if RUN_LLAMA:
    llama_train = make_dataset(splits["train"][:LLAMA_N_TRAIN], llama_tok, add_bos=True)
    e = llama_train[0]
    print("first tokens:", llama_tok.convert_ids_to_tokens(e["input_ids"][:4]),
          "| last:", llama_tok.convert_ids_to_tokens(e["input_ids"][-2:]))
    assert e["input_ids"][0] == llama_tok.bos_token_id and e["input_ids"][1] != llama_tok.bos_token_id
    assert e["input_ids"][-1] == llama_tok.eos_token_id
    llama_rows = eval_rows[:LLAMA_N_EVAL]

    transformers.set_seed(SEED)
    llama = AutoModelForCausalLM.from_pretrained(LLAMA_CKPT, dtype=torch.float16, token=HF_TOKEN, device_map={"": 0})
    print(f"Llama parameters: {sum(p.numel() for p in llama.parameters()):,} | "
          f"GPU memory after loading: {torch.cuda.memory_allocated() / GB:.2f} GB")
    print(llama.model.layers[0].self_attn)

**Extension task A.** The printed attention block shows the projection shapes. Predict the number of LoRA parameters for rank 16 on `q_proj`, `k_proj`, `v_proj`, and `o_proj` in all 16 layers, then complete `llama_lora_config()` (rank 16, `lora_alpha` 32, dropout 0.05, those four target modules, causal LM; no `fan_in_fan_out`, since Llama uses ordinary `nn.Linear` layers).

In [ ]:
def llama_lora_config():
    # TODO: return LoraConfig(...)
    raise NotImplementedError("Extension task A: complete llama_lora_config")


LLAMA_PREDICTED = None   # TODO: your prediction, as an arithmetic expression

The next cell generates with the original Llama, attaches the adapter, and trains. Gradient checkpointing (recomputing activations during the backward pass) keeps memory down; the large vocabulary makes the logits expensive, so the batch is 4 with 2 accumulation steps (an effective batch of 8, as for GPT-2).

In [ ]:
if RUN_LLAMA:
    llama_base_preds, llama_base_stopped = generate_methods(llama, llama_tok, llama_rows, add_bos=True)
    llama_lora = get_peft_model(llama, llama_lora_config())
    llama_lora.print_trainable_parameters()
    assert count_trainable(llama_lora) == LLAMA_PREDICTED, "prediction and peft disagree"
    llama_args = TrainingArguments(
        output_dir=str(OUT / "llama_lora"), per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10, fp16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch", logging_steps=10, save_strategy="no", report_to="none",
        remove_unused_columns=False, seed=SEED)
    llama_val = make_dataset(splits["val"][:100], llama_tok, add_bos=True)
    llama_stats, llama_val_loss = train(llama_lora, llama_args, llama_train, llama_val, "Llama-3.2-1B, LoRA r = 16",
                                        data_collator=lambda b: collate(b, pad_id=llama_tok.pad_token_id))
    llama_lora_preds, llama_lora_stopped = generate_methods(llama_lora, llama_tok, llama_rows, add_bos=True)

The comparison uses the same `LLAMA_N_EVAL` recipes for every model. `sacrebleu` and `rouge_score` tokenize the text themselves, so the scores of models with different tokenizers are comparable.

In [ ]:
if RUN_LLAMA:
    from IPython.display import display
    results.clear()
    k = LLAMA_N_EVAL
    for name, preds, stopped in (("GPT-2 original", base_preds[:k], base_stopped[:k]),
                                 ("GPT-2 full FT", full_preds[:k], full_stopped[:k]),
                                 ("GPT-2 LoRA", lora_preds[:k], lora_stopped[:k]),
                                 ("Llama original", llama_base_preds, llama_base_stopped)):
        evaluate(name, preds, stopped, llama_rows)
    display(evaluate("Llama LoRA", llama_lora_preds, llama_lora_stopped, llama_rows))
    for i in (0, 1):
        print(f"=== {llama_rows[i]['name']} ===\nLLAMA LoRA: {llama_lora_preds[i][:700]}\nGPT-2 LoRA: {lora_preds[i][:700]}\n")

**Extension questions.** (A) Explain your prediction of the LoRA parameter count. Why are the `k_proj` and `v_proj` adapters smaller than the `q_proj` adapter? (B) Llama's vocabulary is about 2.5 times larger than GPT-2's (128,256 vs. 50,257 tokens). Compare the printed token counts and the splits of the Indian words: did the larger vocabulary help much on these recipes? Suggest why. Does the model's tokenizer change BLEU or ROUGE directly? (C) Compare Llama LoRA with GPT-2 LoRA and GPT-2 full fine-tuning. List at least three ways in which this comparison is **not** controlled (think about training data, epochs, model size, rank, evaluation size), and say what you would change to make it fair.

**Your answer — Extension questions:** *(write here)*

## References

1. Jain, K. (2020). [6000+ Indian Food Recipes Dataset](https://data.mendeley.com/datasets/xsphgmmh7b/1). Mendeley Data, V1, doi:10.17632/xsphgmmh7b.1. CC BY 4.0. Recipes from [Archana's Kitchen](https://www.archanaskitchen.com/).
2. Radford, Wu, Child, Luan, Amodei & Sutskever (2019). [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). OpenAI · [GPT-2 on the Hub](https://huggingface.co/openai-community/gpt2).
3. Hu et al. (2021). [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685).
4. Papineni, Roukos, Ward & Zhu (2002). [BLEU: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040/). ACL.
5. Lin (2004). [ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/) · [`rouge_score`](https://github.com/google-research/google-research/tree/master/rouge).
6. Post (2018). [A Call for Clarity in Reporting BLEU Scores](https://aclanthology.org/W18-6319/). WMT · [`sacrebleu`](https://github.com/mjpost/sacrebleu).
7. Llama Team, AI @ Meta (2024). [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783) · [Llama-3.2-1B model card](https://huggingface.co/meta-llama/Llama-3.2-1B).
8. Hugging Face documentation: [PEFT](https://huggingface.co/docs/peft) · [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) · [text generation strategies](https://huggingface.co/docs/transformers/generation_strategies).